In [5]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from pipeline import (
    process_vocab_word
)

import json
import requests
from constants import (
    ANKI_CONNECT_URL    
)

In [6]:
# Function to send the card to Anki
def add_card_to_anki(deck_name, vocab_data):
    """
    Sends the formatted Anki card to Anki using AnkiConnect API.
    """
    note = {
        "deckName": deck_name,
        "modelName": "Basic", 
        "fields": {
            "Front": f"{vocab_data['vocab_word']}\n\n{vocab_data['example_sentence']}",
            "Back": f"{vocab_data['vocab_translation']}\n\n{vocab_data['example_sentence_translation']}",
        },
        "audio": [
            {"url": vocab_data["vocab_audio"], "filename": "vocab_word.mp3", "fields": ["Front"]},
            {"url": vocab_data["example_sentence_translation_audio"], "filename": "example_sentence.mp3", "fields": ["Back"]}
        ]
    }
    
    payload = {"action": "addNote", "version": 6, "params": {"note": note}}
    
    response = requests.post(ANKI_CONNECT_URL, json=payload).json()
    return response

In [7]:
def request(action, **params):
    return {'action': action, 'params': params, 'version': 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    
    response_json = response.json()
    
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    
    return response_json['result']

In [8]:
invoke('deckNames')

['* Navajo',
 '5000 Most Common French Words',
 '5000 Most Common French Words::[1] Main Course',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::I) French to English (Start here)',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::II) English to French',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::I) French to English',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::II) English to French',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 1: Most Frequent Conjs. Come First',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 2: One Verb a